# 03. Indexing, Slicing & Selection Mechanics: Beginner Guide

### 📌 Overview & Architectural Context
Welcome to **03. Indexing, Slicing & Selection Mechanics**. Mastering data selection is vital for writing performant, bug-free Pandas workflows. This notebook breaks down the distinctions between label-based indexing (`.loc[]`), coordinate positional slicing (`.iloc[]`), fast scalar lookups (`.at[]`, `.iat[]`), vectorized Boolean filtering, dynamic C-speed query expressions (`.query()`), and copy-vs-view memory semantics to eliminate `SettingWithCopyWarning`.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Label Selection: `DataFrame.loc[]`
- [x] 🔹 Fast Scalar Label Lookup: `DataFrame.at[]`
- [x] 🔹 Integer Position Selection: `DataFrame.iloc[]`
- [x] 🔹 Fast Scalar Position Lookup: `DataFrame.iat[]`
- [x] 🔹 Vectorized Boolean Mask Filtering: `DataFrame[condition]`
- [x] 🔹 Dynamic Expression Filtering: `DataFrame.query()`
- [x] 🔹 Conditional Value Replacement: `DataFrame.where()`
- [x] 🔹 Inverse Conditional Masking: `DataFrame.mask()`


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import sqlite3
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head(2))

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns
  transaction_id customer_id merchant_id  transaction_amount card_type  \
0       TX109326      C55082       M3549              607.78      Visa   
1       TX106376      C76616       M3068             1819.11      Visa   

  transaction_status device_type  account_age_months     transaction_date  \
0           Reversed      Mobile                   8  2026-02-17 08:28:57   
1            Pending         POS                  28          03-Jan-2025   

  region  is_fraud  
0  North         0  
1   West         1  


### 🔹 Label Selection: `DataFrame.loc[]`
- **What it does:** Accesses a group of rows and columns by label(s) or a boolean array.
- **Syntax:** `DataFrame.loc[row_indexer, column_indexer]`
  - **Parameters:**
    - `row_indexer` (*scalar, slice, list, or boolean mask*): Row identifier(s).
  - **Optional Parameters:**
    - `col_indexer` (*scalar, slice, list, or boolean mask*): Column identifier(s).
- **Key Note:** Label slicing with `.loc[]` includes **both** the start and stop boundaries (unlike integer slicing which excludes the stop bound).
- **Dataset Application & Code Demonstration:** Selects rows 0 through 5 and isolates fintech columns `transaction_id`, `transaction_amount`, and `card_type`.


In [2]:
print('df.loc Selection (Rows 0-3):\n', df.loc[0:3, ['transaction_id', 'transaction_amount', 'card_type']])

df.loc Selection (Rows 0-3):
   transaction_id  transaction_amount card_type
0       TX109326              607.78      Visa
1       TX106376             1819.11      Visa
2       TX103301               64.08      Visa
3       TX110701             1025.73      Amex


### 🔹 Fast Scalar Label Lookup: `DataFrame.at[]`
- **What it does:** Accesses a single scalar value for a row/column pair by label, providing maximum speed by bypassing Series wrapper overhead.
- **Syntax:** `DataFrame.at[row_label, column_label]`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** `.at[]` is optimized strictly for scalar lookups and is 3-5x faster than `.loc[]` in tight iteration loops.
- **Dataset Application & Code Demonstration:** Directly retrieves a single scalar `transaction_amount` at row index 0.


In [3]:
print('Fast Scalar (df.at[0, amount]):', df.at[0, 'transaction_amount'])

Fast Scalar (df.at[0, amount]): 607.78


### 🔹 Integer Position Selection: `DataFrame.iloc[]`
- **What it does:** Purely integer-location based indexing for selection by 0-indexed position coordinates.
- **Syntax:** `DataFrame.iloc[row_indexer, column_indexer]`
  - **Parameters:**
    - `row_indexer` (*scalar, slice, list, or boolean mask*): Row identifier(s).
  - **Optional Parameters:**
    - `col_indexer` (*scalar, slice, list, or boolean mask*): Column identifier(s).
- **Key Note:** `.iloc[]` adheres to standard Python 0-indexed slicing semantics: the start index is included and the stop index is excluded.
- **Dataset Application & Code Demonstration:** Extracts the first 4 rows and first 3 columns strictly using integer coordinate offsets.


In [4]:
print('df.iloc Selection (First 4 rows, first 3 cols):\n', df.iloc[0:4, 0:3])

df.iloc Selection (First 4 rows, first 3 cols):
   transaction_id customer_id merchant_id
0       TX109326      C55082       M3549
1       TX106376      C76616       M3068
2       TX103301      C65296       M3352
3       TX110701      C42098       M5807


### 🔹 Fast Scalar Position Lookup: `DataFrame.iat[]`
- **What it does:** Accesses a single scalar value for a row/column pair by integer position coordinates.
- **Syntax:** `DataFrame.iat[row_position, column_position]`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Use `.iat[]` in time-critical algorithms where position coordinates are known, avoiding index label hash lookups.
- **Dataset Application & Code Demonstration:** Directly retrieves the scalar value at row offset 0 and column offset 3.


In [5]:
print('Fast Position Scalar (df.iat[0, 3]):', df.iat[0, 3])

Fast Position Scalar (df.iat[0, 3]): 607.78


### 🔹 Vectorized Boolean Mask Filtering: `DataFrame[condition]`
- **What it does:** Filters DataFrame rows using vectorized boolean conditions combined with bitwise logical operators (`&`, `|`, `~`).
- **Syntax:** `DataFrame[condition]`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** When combining multiple boolean criteria in pandas, each condition must be wrapped in parentheses `()` to enforce correct bitwise precedence.
- **Dataset Application & Code Demonstration:** Applies Vectorized Boolean Mask Filtering on fintech records using columns `card_type`, `is_fraud`, `region`, `transaction_amount`, `transaction_id` to demonstrate real-world execution.


In [6]:
fraud_high_val = df[(df['transaction_amount'] > 500.0) & (df['is_fraud'] == 1)]
print(f'High-Value Fraud Transactions Found: {len(fraud_high_val)}')
print(fraud_high_val[['transaction_id', 'transaction_amount', 'card_type', 'region']].head(3))

High-Value Fraud Transactions Found: 1572
   transaction_id  transaction_amount   card_type region
1        TX106376             1819.11        Visa   West
18       TX111101             1998.80  MasterCard   West
19       TX107785             1806.44        Visa  North


### 🔹 Dynamic Expression Filtering: `DataFrame.query()`
- **What it does:** Filters DataFrame rows with a boolean expression evaluated as a dynamic string using the numexpr engine.
- **Syntax:** `DataFrame.query(expr, *, inplace=False, **kwargs)`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Variables from the surrounding Python scope can be referenced inside the query string by prefixing them with the `@` character.
- **Dataset Application & Code Demonstration:** Queries records where `transaction_amount > @threshold` and `is_fraud == 1` using local variable binding.


In [7]:
threshold = 800.0
queried_fraud = df.query('transaction_amount > @threshold and is_fraud == 1')
print('Queried Fraud Count:', len(queried_fraud))

Queried Fraud Count: 1554


### 🔹 Conditional Value Replacement: `DataFrame.where()`
- **What it does:** Replaces values where the condition evaluates to False, retaining values where condition is True.
- **Syntax:** `DataFrame.where(cond, other=_NoDefault.no_default, *, inplace=False, axis=None, level=None)`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** `.where()` keeps matching values and replaces non-matching ones, whereas `.mask()` is the inverse (replaces matching values).
- **Dataset Application & Code Demonstration:** Preserves `transaction_amount` values greater than 100 and replaces sub-threshold values with 0.0.


In [8]:
print('Where (> 100):\n', df['transaction_amount'].where(df['transaction_amount'] > 100, other=0.0).head())

Where (> 100):
 0     607.78
1    1819.11
2       0.00
3    1025.73
4     772.74
Name: transaction_amount, dtype: float64


### 🔹 Inverse Conditional Masking: `DataFrame.mask()`
- **What it does:** Replaces values where the condition evaluates to True (exact inverse of `df.where()`).
- **Syntax:** `DataFrame.mask(cond, other=_NoDefault.no_default, *, inplace=False, axis=None, level=None)`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Use `.mask()` to quickly censor or nullify sensitive values matching a condition.
- **Dataset Application & Code Demonstration:** Masks anomalous high-spend transactions exceeding 900 by setting them to `NaN`.


In [9]:
print('Masked (> 1000 capped):\n', df['transaction_amount'].mask(df['transaction_amount'] > 1000, other=1000.0).head())

Masked (> 1000 capped):
 0     607.78
1    1000.00
2      64.08
3    1000.00
4     772.74
Name: transaction_amount, dtype: float64


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: In-Place Mutation vs SettingWithCopyWarning
- **Objective:** Q1: In-Place Mutation vs SettingWithCopyWarning
- **Approach:** Correctly assign risk flags to high-value transactions using `df.loc` to avoid SettingWithCopyWarning.
- **Syntax:** `df.loc[df['transaction_amount'] > 500, 'risk_tier'] = 'High'`

In [10]:
df_copy = df.head(10).copy()
df_copy.loc[df_copy['transaction_amount'] > 200, 'risk_tier'] = 'High'
print(df_copy[['transaction_id', 'transaction_amount', 'risk_tier']])

  transaction_id  transaction_amount risk_tier
0       TX109326              607.78      High
1       TX106376             1819.11      High
2       TX103301               64.08       NaN
3       TX110701             1025.73      High
4       TX103284              772.74      High
5       TX104210              198.47       NaN
6       TX106427              217.23      High
7       TX104105             1070.66      High
8       TX114893                 NaN       NaN
9       TX114584             1320.66      High
